# 期限结构输入

本单元把真实合约的采样值与合约最后有效日期连接，输出每个采样点、每个合约的数值与剩余期限，供 NSS 拟合和网格校准调用。输入值由上游选择，原样保留；本单元不取 log、不指数还原，也不因数值为零或负数而排除。模型系数的经济约束由后续拟合调用明确指定。

## 剩余期限

终点为合约最后有效日期 `delist_date` 的自然日末：用上海时区的次日 00:00 表示 `terminal_at`，即 `delist_date + 1 day` 的日界。最后有效日的整天计入期限；不取该日零点、不截成整数天，也不需要到期日的 Session 行。

$$\mathrm{maturity\_days}=\frac{\mathrm{terminal\_at}-\mathrm{sample\_at}}{86400\ \mathrm{seconds}}.$$

按自然经过时间计算，包含休市间隔并保留小数天。零期限保留给下游基函数的极限值处理；负期限标记 `excluded`。缺少合约最后有效日期时标记 `failed / missing_delist_date`。只要合约元信息中存在 `delist_date`，包括未来到期的合约，都可构造日末终点。

分钟内插值得到的 `sample_at` 是采样位置，实际报价信息在 `bar_at` 才完整可得。迭代输出用 `available_at` 保留这一区别，不能把分钟内插值解释为在 `sample_at` 当时可得的价格。

## 对外函数

- `build_contract_terminals(contract_calendar_df)`：从已读取的日历构造每个合约的终点及状态。
- `prepare_term_structure_inputs(sampled_value_df, contract_terminal_df, value_col=...)`：逐行连接，输出输入值、期限、状态和原因，保持合约和值的对应关系。
- `iter_term_structure_curves(term_structure_df)`：逐采样点返回按期限排列的有效合约、期限和值，并保留该点全部行的诊断。有效数组可以为空；不设合约数量门槛。

定义及导入代理见本单元同名 `.py`。研究目录规则见 [研究 AGENTS](../../AGENTS.md)，上游输入为 [采样执行](../b03_Sampling/c02_SamplingExecution.ipynb) 的全合约长表。下面的真实 demo 使用 RB 的完整固定分界序列；它验收输入构造，不执行 NSS 拟合或 λ 寻优。

In [1]:
from pathlib import Path
import sys

project_markers = [".git", ".env", "config/settings.py"]
current_path = Path.cwd().resolve()
for candidate_root in [current_path, *current_path.parents]:
    if all((candidate_root / marker).exists() for marker in project_markers):
        sys.path.insert(0, str(candidate_root))
        from config.settings import settings
        break
else:
    raise RuntimeError("未找到包含 .git、.env、config/settings.py 的项目根目录")


In [2]:
from __future__ import annotations

import numpy as np
import pandas as pd
import pyarrow as pa
from config.data_contracts import FUTURES_CONTRACT_CALENDAR_SCHEMA, FUTURES_MINUTE_SCHEMA, pandas_to_arrow, arrow_to_pandas
from R04_Research.a01_Methods.b03_Sampling.c01_SamplingPositions import SAMPLING_POSITION_SCHEMA

TERM_STRUCTURE_INPUTS_VERSION = "0.2.0"
TERM_STRUCTURE_TIMEZONE = "Asia/Shanghai"
TERM_STRUCTURE_TERMINAL_RULE = "end_of_delist_date"
CONTRACT_CALENDAR_TABLE_NAME = FUTURES_CONTRACT_CALENDAR_SCHEMA.metadata[b"table_name"].decode()
CONTRACT_CALENDAR_PARTITION_COLUMNS = FUTURES_CONTRACT_CALENDAR_SCHEMA.metadata[b"partition_columns"].decode().split(",")
CONTRACT_CALENDAR_PRIMARY_KEY = FUTURES_CONTRACT_CALENDAR_SCHEMA.metadata[b"primary_key"].decode().split(",")
CONTRACT_ENTITY_COLUMNS = ["exchange_code", "underlying_code", "contract_code"]

CONTRACT_TERMINAL_SCHEMA = pa.schema([
    *[FUTURES_CONTRACT_CALENDAR_SCHEMA.field(name) for name in CONTRACT_ENTITY_COLUMNS],
    FUTURES_CONTRACT_CALENDAR_SCHEMA.field("delist_date").with_nullable(True),
    pa.field("terminal_at", pa.timestamp("us", tz=TERM_STRUCTURE_TIMEZONE), metadata={
        b"field_name_zh": "最后有效日的自然日末边界".encode(),
        b"description_zh": "Asia/Shanghai 时区 delist_date 次日 00:00，表示最后有效日完整结束；缺 delist_date 时为空".encode()}),
    pa.field("terminal_status", pa.string(), nullable=False, metadata={b"field_name_zh": "合约终点状态".encode()}),
    pa.field("terminal_reason", pa.string(), nullable=False, metadata={b"field_name_zh": "合约终点状态原因".encode()}),
], metadata={b"method_name": b"term_structure_inputs", b"method_version": TERM_STRUCTURE_INPUTS_VERSION.encode(),
    b"primary_key": b"exchange_code,underlying_code,contract_code", b"timezone": TERM_STRUCTURE_TIMEZONE.encode(), b"terminal_rule": TERM_STRUCTURE_TERMINAL_RULE.encode()})

TERM_STRUCTURE_INPUT_SCHEMA = pa.schema([
    *SAMPLING_POSITION_SCHEMA,
    *[FUTURES_MINUTE_SCHEMA.field(name) for name in CONTRACT_ENTITY_COLUMNS],
    pa.field("is_main", pa.bool_(), nullable=False),
    pa.field("input_value", pa.float64(), metadata={b"field_name_zh": "上游指定的输入值".encode(),
        b"description_zh": "原样采用 value_col；有限零值及负值保留；非有限值落盘为空并记录失败".encode()}),
    FUTURES_CONTRACT_CALENDAR_SCHEMA.field("delist_date").with_nullable(True),
    CONTRACT_TERMINAL_SCHEMA.field("terminal_at"),
    pa.field("maturity_days", pa.float64(), metadata={b"field_name_zh": "自然经过时间剩余天数".encode(),
        b"unit_zh": "自然日，可含小数".encode()}),
    pa.field("input_status", pa.string(), nullable=False, metadata={b"field_name_zh": "期限结构输入状态".encode()}),
    pa.field("input_reason", pa.string(), nullable=False, metadata={b"field_name_zh": "期限结构输入状态原因".encode()}),
], metadata={b"method_name": b"term_structure_inputs", b"method_version": TERM_STRUCTURE_INPUTS_VERSION.encode(),
    b"primary_key": b"sample_id,contract_code", b"timezone": TERM_STRUCTURE_TIMEZONE.encode(), b"terminal_rule": TERM_STRUCTURE_TERMINAL_RULE.encode(),
    b"input_domain": b"upstream supplied; no internal transformation",
    b"maturity_definition": b"(terminal_at - sample_at).total_seconds() / 86400"})


In [3]:
from R02_Market_Data.a01_Collection.b00_03_notebook_schema_browser import display_schema_metadata
display_schema_metadata([FUTURES_CONTRACT_CALENDAR_SCHEMA])


In [4]:
# 纯展示标签不参与生成身份，silver 中文直接来自当前权威字段 metadata。
TERM_STRUCTURE_COLUMN_LABELS = {
    field.name: f"{field.name}（{field.metadata[b'field_name_zh'].decode()}）"
    for field in FUTURES_CONTRACT_CALENDAR_SCHEMA
}
TERM_STRUCTURE_COLUMN_LABELS.update({
    name: f"{name}（{meaning}）" for name, meaning in {
        "sample_id": "采样编号", "t": "采样分界", "sample_side": "采样侧", "sample_number": "侧内采样编号",
        "source_minute_index": "源分钟坐标", "sample_at": "采样时刻", "bar_at": "源分钟结束时刻",
        "main_contract_code": "主力合约", "interpolation_fraction": "分钟内位置比例", "is_main": "是否当时主力",
        "input_value": "上游指定的输入值", "terminal_at": "最后有效日的自然日末边界",
        "terminal_status": "合约终点状态", "terminal_reason": "合约终点状态原因",
        "maturity_days": "自然经过时间剩余天数", "input_status": "期限结构输入状态", "input_reason": "期限结构输入状态原因",
    }.items()
})


## 合约终点

日历读取属于调用方，函数不调用 API 或读取数据湖。输入只需 `exchange_code`、`underlying_code`、`contract_code`、`delist_date`；可以直接提供合约元信息，也可以提供包含这些列的多交易日、多 Session 日历。函数按合约去重，使用上海时区的 `delist_date` 次日 00:00 构造日末终点，不读取 `trading_date` 或 `session_end_at`。缺日期的合约保留失败记录；同一合约有互相冲突的最后有效日期时明确报错。正式 silver 的业务契约由上游保证，本单元仅校验形成唯一连接所必需的条件。


In [5]:
def build_contract_terminals(contract_calendar_df: pd.DataFrame | pa.Table) -> pd.DataFrame:
    '''以合约 delist_date 次日上海零点表示最后有效日末；不依赖 Session。'''
    calendar_columns = [*CONTRACT_ENTITY_COLUMNS, "delist_date"]
    calendar_df = (contract_calendar_df.select(calendar_columns).to_pandas()
                   if isinstance(contract_calendar_df, pa.Table) else contract_calendar_df[calendar_columns].copy())
    calendar_df["delist_date"] = pd.to_datetime(calendar_df["delist_date"]).dt.date
    contract_terminal_df = calendar_df[calendar_columns].drop_duplicates().copy()
    if contract_terminal_df.duplicated(CONTRACT_ENTITY_COLUMNS).any():
        raise ValueError("同一合约的 delist_date 不唯一，无法形成终点连接")
    terminal_dates = pd.DatetimeIndex(pd.to_datetime(contract_terminal_df["delist_date"]))
    contract_terminal_df["terminal_at"] = (
        terminal_dates.tz_localize(TERM_STRUCTURE_TIMEZONE) + pd.DateOffset(days=1))
    missing_terminal = contract_terminal_df["terminal_at"].isna()
    contract_terminal_df["terminal_status"] = np.where(missing_terminal, "failed", "ok")
    contract_terminal_df["terminal_reason"] = np.where(missing_terminal, "missing_delist_date", "")
    return arrow_to_pandas(pandas_to_arrow(contract_terminal_df, CONTRACT_TERMINAL_SCHEMA), CONTRACT_TERMINAL_SCHEMA)


## 逐采样点输入

输出长表保持上游全部合约行。`ok` 行具有有限输入值、已知终点及非负剩余期限；`failed` 行保留缺失终点或非有限输入的原因；`excluded` 行表示已经越过合约终点。多个原因以分号连接。非有限输入转为 Arrow null，有限输入不变，负期限保留原计算结果以便诊断。

计算不设置合约数量门槛，不固定合约槽位，也不对价格或期限独立压缩。一个采样点可以有 11 个、其他数量乃至零个有效合约，均由后续拟合正常求解并记录结果。

In [6]:
def prepare_term_structure_inputs(sampled_value_df: pd.DataFrame | pa.Table,
                                 contract_terminal_df: pd.DataFrame | pa.Table, *, value_col: str) -> pd.DataFrame:
    '''原样接收指定数值列，按真实合约连接期限，保留每行的状态和原因。'''
    sample_columns = [*SAMPLING_POSITION_SCHEMA.names, *CONTRACT_ENTITY_COLUMNS, "is_main"]
    sampled_df = (sampled_value_df.select([*sample_columns, value_col]).to_pandas()
                  if isinstance(sampled_value_df, pa.Table) else sampled_value_df[[*sample_columns, value_col]].copy())
    sampled_df = sampled_df.reset_index(drop=True)
    if sampled_df.duplicated(["sample_id", "contract_code"]).any():
        raise ValueError("sample_id、contract_code 必须唯一，不能形成重复合约输入")
    sampled_df["input_value"] = sampled_df[value_col].to_numpy(dtype=np.float64, na_value=np.nan)
    sampled_df = sampled_df[[*sample_columns, "input_value"]]
    terminal_df = (contract_terminal_df.to_pandas() if isinstance(contract_terminal_df, pa.Table)
                   else contract_terminal_df.copy())
    term_structure_df = sampled_df.merge(terminal_df, on=CONTRACT_ENTITY_COLUMNS, how="left", sort=False, validate="many_to_one")
    sample_times = pd.DatetimeIndex(term_structure_df["sample_at"])
    terminal_times = pd.DatetimeIndex(term_structure_df["terminal_at"])
    if ((sample_times.notna().any() and sample_times.tz is None)
            or (terminal_times.notna().any() and terminal_times.tz is None)):
        raise ValueError("sample_at 与 terminal_at 必须包含明确时区")
    term_structure_df["sample_at"] = (sample_times.tz_convert(TERM_STRUCTURE_TIMEZONE) if sample_times.tz
                                      else sample_times.tz_localize(TERM_STRUCTURE_TIMEZONE))
    term_structure_df["terminal_at"] = (terminal_times.tz_convert(TERM_STRUCTURE_TIMEZONE) if terminal_times.tz
                                        else terminal_times.tz_localize(TERM_STRUCTURE_TIMEZONE))
    term_structure_df["maturity_days"] = (term_structure_df["terminal_at"] - term_structure_df["sample_at"]).dt.total_seconds() / 86400.0

    reasons = np.full(len(term_structure_df), "", dtype=object)
    failures = np.zeros(len(term_structure_df), dtype=bool)
    invalid_conditions = [
        (term_structure_df["terminal_status"].isna().to_numpy(), "contract_not_in_terminal_table"),
        (term_structure_df["terminal_status"].notna().to_numpy() & term_structure_df["terminal_at"].isna().to_numpy(), "missing_delist_date"),
        (term_structure_df["sample_at"].isna().to_numpy(), "missing_sample_at"),
        (~np.isfinite(term_structure_df["input_value"].to_numpy(dtype=float)), "nonfinite_input_value"),
    ]
    for invalid, reason in invalid_conditions:
        reasons[invalid] = np.where(reasons[invalid] == "", reason, reasons[invalid] + ";" + reason)
        failures |= invalid
    expired = term_structure_df["maturity_days"].lt(0).fillna(False).to_numpy(dtype=bool)
    reasons[expired] = np.where(reasons[expired] == "", "past_terminal_at", reasons[expired] + ";past_terminal_at")
    term_structure_df["input_status"] = np.where(failures, "failed", np.where(expired, "excluded", "ok"))
    term_structure_df["input_reason"] = reasons
    term_structure_df.loc[~np.isfinite(term_structure_df["input_value"]), "input_value"] = np.nan
    return arrow_to_pandas(pandas_to_arrow(term_structure_df[TERM_STRUCTURE_INPUT_SCHEMA.names], TERM_STRUCTURE_INPUT_SCHEMA), TERM_STRUCTURE_INPUT_SCHEMA)


def iter_term_structure_curves(term_structure_df: pd.DataFrame | pa.Table):
    '''逐采样点输出成对的合约、期限和值；同时保留全部合约行的诊断。'''
    inputs_df = term_structure_df.to_pandas() if isinstance(term_structure_df, pa.Table) else term_structure_df
    for sample_id, curve_rows_df in inputs_df.groupby("sample_id", sort=False):
        sample_metadata_df = curve_rows_df[SAMPLING_POSITION_SCHEMA.names].drop_duplicates()
        if len(sample_metadata_df) != 1:
            raise ValueError(f"采样点 {sample_id} 的公共位置字段不一致")
        usable_rows_df = curve_rows_df.loc[curve_rows_df["input_status"].eq("ok")].sort_values(["maturity_days", "contract_code"])
        metadata = sample_metadata_df.iloc[0].to_dict()
        yield {**metadata, "available_at": curve_rows_df["bar_at"].max(),
               "contract_code": usable_rows_df["contract_code"].to_numpy(dtype=str),
               "maturity_days": usable_rows_df["maturity_days"].to_numpy(dtype=np.float64),
               "input_value": usable_rows_df["input_value"].to_numpy(dtype=np.float64),
               "row_status_df": curve_rows_df.copy()}


In [7]:
def load_nss_demo_config(config_path) -> dict:
    """显式读取三个 NSS 单元共用参数；不计算、不保存、不连接云。"""
    from pathlib import Path
    from copy import deepcopy
    import yaml
    config = yaml.safe_load(Path(config_path).read_text(encoding='utf8'))
    if set(config) != {'scope','input','trial','nss','cloud','plot'}:
        raise ValueError('NSS 配置顶层字段不匹配')
    if config['trial']['save_data'] is not False or config['trial']['selection'] != 'first_points_per_side':
        raise ValueError('试算必须为每侧完整采样点，禁止保存数据文件')
    for key in ('train_points','test_points'):
        if isinstance(config['trial'][key],bool) or not isinstance(config['trial'][key],int) or config['trial'][key] <= 0:
            raise ValueError('试算采样点数必须为正整数')
    if config['input']['transform'] != 'identity' or config['input']['terminal_rule'] != TERM_STRUCTURE_TERMINAL_RULE:
        raise ValueError('输入变换由上游决定，期限终点须为自然日末')
    if config['scope']['timezone'] != TERM_STRUCTURE_TIMEZONE:
        raise ValueError('NSS 真实输入必须采用 Asia/Shanghai')
    if config['nss']['selection']['methods'] != ['min_error','legacy_frequency','legacy_error_weighted']:
        raise ValueError('三个已定义的 λ 选择方法须并列保留')
    if config['nss']['selection']['default_method'] != 'min_error':
        raise ValueError('正式最小误差定义须为默认选择')
    for section,key in [('nss','point_chunk_size'),('cloud','worker_processes')]:
        value=config[section][key]
        if isinstance(value,bool) or not isinstance(value,int) or value < 1:
            raise ValueError(key+' 必须为正整数')
    if config['cloud']['instance_count'] != 1 or config['cloud']['network'] != 'PUBLIC':
        raise ValueError('首版完整 NSS demo 使用一个 PUBLIC DLC 节点')
    return deepcopy(config)

def load_term_structure_demo_source(project_root, config, *, trial=True) -> dict:
    """精确读取已验收采样和只读日历。trial 每侧截取完整点；不写数据文件。"""
    from pathlib import Path
    import hashlib,json
    import pyarrow.dataset as ds
    import pyarrow.parquet as pq
    from config.settings import settings
    from R04_Research.a01_Methods.b02_MainContinuousAdjustment.c01_MainContinuousAdjustment import arrow_content_sha256
    from R04_Research.a01_Methods.b03_Sampling.c02_SamplingExecution import SAMPLED_CLOSE_SCHEMA
    project_root=Path(project_root)
    scope,parameters=config['scope'],config['input']
    range_token=f"train{scope['train']['start'].replace('-','')}-{scope['train']['end'].replace('-','')}_test{scope['test']['start'].replace('-','')}-{scope['test']['end'].replace('-','')}"
    sampling_token=f"N{parameters['N']}_{parameters['axis']}_{parameters['quantity_scope']}_{parameters['sampling_method']}_{parameters['execution_mode']}"
    sampling_dir=project_root/'R04_Research/a01_Methods/b03_Sampling'
    input_path=sampling_dir/'c02_SamplingExecution_DemoData'/f"c02_SamplingExecution_{scope['underlying_code']}_{range_token}_{sampling_token}_demo.parquet"
    sampled_table=pq.read_table(input_path)
    identity=json.loads(sampled_table.schema.metadata[b'demo_identity'])
    expected_parameters={key:parameters[key] for key in ['N','axis','quantity_scope','sampling_method','execution_mode','t']}
    expected_parameters['K']=None
    expected_parameters['test_start_date']=scope['test']['start']
    if identity['parameters'] != expected_parameters:
        raise ValueError('上游采样配置不匹配')
    upstream_scope=identity['scope']
    # 请求起点可能是休市日；上游训练范围记录实际首个交易日。
    if (upstream_scope['mapping_read'] != {'start':scope['train']['start'],'end':scope['test']['end']}
        or upstream_scope['training_trading_dates']['end'] != scope['train']['end']
        or not scope['train']['start'] <= upstream_scope['training_trading_dates']['start'] <= scope['train']['end']
        or upstream_scope['test_trading_dates'] != scope['test']):
        raise ValueError('上游训练／测试范围不匹配')
    if identity['upstream_demo_identity']['underlying_code'] != scope['underlying_code']:
        raise ValueError('上游品种不匹配')
    for role,name in [('sampling_positions','c01_SamplingPositions'),('sampling_execution','c02_SamplingExecution')]:
        book=json.loads((sampling_dir/(name+'.ipynb')).read_text(encoding='utf8'))
        source='\n\n'.join(''.join(c['source']) for c in book['cells'] if c['cell_type']=='code'
            and {'export','demo-dependency'}.issubset(c.get('metadata',{}).get('tags',[])))
        if hashlib.sha256(source.encode()).hexdigest() != identity['dependency_method_source_sha256'][role]:
            raise ValueError('上游采样生成定义已变化，须先重算采样')
    for role,relative in [('notebook_loader','R04_Research/a00_notebook_loader.py'),('data_contracts','config/data_contracts.py')]:
        if hashlib.sha256((project_root/relative).read_bytes()).hexdigest() != identity['dependency_source_sha256'][role]:
            raise ValueError('上游转换／加载依赖已变化')
    if arrow_content_sha256(sampled_table,SAMPLED_CLOSE_SCHEMA,identity['output_sort_keys']) != identity['output_content_sha256']:
        raise ValueError('上游采样内容摘要不一致')
    sampled_df=sampled_table.to_pandas()
    training_dates=sampled_df.loc[sampled_df['sample_side'].eq('train'),'trading_date']
    if training_dates.empty or str(training_dates.min()) != upstream_scope['training_trading_dates']['start']:
        raise ValueError('上游实际训练起点与采样内容不一致')
    if trial:
        selected_ids=[]
        for side,limit in [('train',config['trial']['train_points']),('test',config['trial']['test_points'])]:
            selected_ids.extend(sampled_df.loc[sampled_df['sample_side'].eq(side),'sample_id'].drop_duplicates().iloc[:limit].tolist())
        sampled_df=sampled_df.loc[sampled_df['sample_id'].isin(selected_ids)].reset_index(drop=True)
    calendar_schema=FUTURES_CONTRACT_CALENDAR_SCHEMA
    calendar_path=Path(settings.futures_lake_root)/'silver'/CONTRACT_CALENDAR_TABLE_NAME
    calendar_dataset=ds.dataset(calendar_path,format='parquet',schema=calendar_schema,
        partitioning=ds.partitioning(pa.schema([calendar_schema.field(name) for name in CONTRACT_CALENDAR_PARTITION_COLUMNS]),flavor='hive'))
    snapshots=sampled_df.groupby('contract_code',as_index=False)['trading_date'].max()
    snapshot_filter=ds.scalar(False)
    for day,rows in snapshots.groupby('trading_date',sort=True):
        snapshot_filter=snapshot_filter|((ds.field('trading_date')==day)&ds.field('contract_code').isin(rows['contract_code'].tolist()))
    calendar_filter=(ds.field('exchange_code')==scope['exchange_code'])&(ds.field('underlying_code')==scope['underlying_code'])&snapshot_filter
    calendar_table=calendar_dataset.to_table(filter=calendar_filter).sort_by([(name,'ascending') for name in CONTRACT_CALENDAR_PRIMARY_KEY])
    return dict(sampled_df=sampled_df,sampled_table=sampled_table if not trial else None,
        calendar_table=calendar_table,calendar_df=arrow_to_pandas(calendar_table,calendar_schema),
        upstream_identity=identity,upstream_path=input_path,range_token=range_token,sampling_token=sampling_token,
        calendar_input=dict(table_name=CONTRACT_CALENDAR_TABLE_NAME,filter=str(calendar_filter),rows=calendar_table.num_rows,
            content_sha256=arrow_content_sha256(calendar_table,calendar_schema,[(name,'ascending') for name in CONTRACT_CALENDAR_PRIMARY_KEY])))


## 默认真实小样本试算（不保存数据文件）

参数由同主题 `c03_NSSCalibration.yaml` 统一定义：RB、N=15、gk_money、all_contracts、log_interpolate、fixed。每分钟采样轴增量为主力合约 GK 波动率 σ 乘该品种全部合约成交额，采样单位为完整训练期平均分钟增量的 15 倍，测试期固定此单位；N 不表示固定每 15 分钟采样。完整请求训练 2010—2024，测试 2025—2026-09-30。当前 Notebook 仅计算每侧指定的少量完整采样点，默认 32 个训练点与 16 个测试点，保留每点全部真实合约。不读取旧 NSS demo 作为试算输入，不写 Parquet、CSV 或其他计算结果文件；表格和图形仅保留在 Notebook 内。全区间九份产物由 c03 的显式云入口计算及回收。

In [8]:
from IPython.display import display
demo_config_path=candidate_root/'R04_Research/a01_Methods/b05_TermStructureModeling/c03_NSSCalibration.yaml'
demo_config=load_nss_demo_config(demo_config_path)
demo_source=load_term_structure_demo_source(candidate_root,demo_config,trial=True)
demo_terminal_df=build_contract_terminals(demo_source['calendar_df'])
demo_inputs_df=prepare_term_structure_inputs(demo_source['sampled_df'],demo_terminal_df,value_col=demo_config['input']['value_col'])
assert len(demo_inputs_df)==len(demo_source['sampled_df'])
assert demo_inputs_df['sample_id'].nunique()<=demo_config['trial']['train_points']+demo_config['trial']['test_points']
print(f"真实小样本：{demo_inputs_df['sample_id'].nunique()} 点、{len(demo_inputs_df)} 合约行；计算结果不写数据文件")


真实小样本：48 点、576 合约行；计算结果不写数据文件


In [9]:
expected_days=(pd.to_datetime(demo_inputs_df['terminal_at'])-pd.to_datetime(demo_inputs_df['sample_at'])).dt.total_seconds()/86400
np.testing.assert_allclose(demo_inputs_df['maturity_days'].astype(float),expected_days.astype(float),equal_nan=True)
for curve in iter_term_structure_curves(demo_inputs_df):
    expected=curve['row_status_df'].loc[curve['row_status_df']['input_status'].eq('ok')].sort_values(['maturity_days','contract_code'])
    np.testing.assert_array_equal(curve['contract_code'],expected['contract_code'].to_numpy(dtype=str))
    np.testing.assert_allclose(curve['input_value'],expected['input_value'].to_numpy(dtype=float))
display(demo_inputs_df.tail(8).rename(columns=TERM_STRUCTURE_COLUMN_LABELS).rename_axis('row（真实试算末端行）'))
print('c01 真实试算的期限、合约和值共同排列核对通过；未保存计算数据文件')


c01 真实试算的期限、合约和值共同排列核对通过；未保存计算数据文件


,sample_id（采样编号）,t（采样分界）,sample_side（采样侧）,sample_number（侧内采样编号）,source_minute_index（源分钟坐标）,sample_at（采样时刻）,trading_date（Session 归属的交易日）,bar_at（源分钟结束时刻）,main_contract_code（主力合约）,interpolation_fraction（分钟内位置比例）,exchange_code（合约所属聚宽交易所代码）,underlying_code（合约所属期货品种代码）,contract_code（聚宽标准固定月份合约代码）,is_main（是否当时主力）,input_value（上游指定的输入值）,delist_date（合约最后有效日期/退市日期）,terminal_at（最后有效日的自然日末边界）,maturity_days（自然经过时间剩余天数）,input_status（期限结构输入状态）,input_reason（期限结构输入状态原因）
row（真实试算末端行）,,,,,,,,,,,,,,,,,,,,
568,75622,2024-12-31 15:00:00+08:00,test,16,1134615,2025-01-03 14:30:50.286774+08:00,2025-01-03,2025-01-03 14:31:00+08:00,RB2505.XSGE,0.838113,XSGE,RB,RB2505.XSGE,True,3271.808918,2025-05-15,2025-05-16 00:00:00+08:00,132.395251,ok,
569,75622,2024-12-31 15:00:00+08:00,test,16,1134615,2025-01-03 14:30:50.286774+08:00,2025-01-03,2025-01-03 14:31:00+08:00,RB2505.XSGE,0.838113,XSGE,RB,RB2506.XSGE,False,3272.0,2025-06-16,2025-06-17 00:00:00+08:00,164.395251,ok,
570,75622,2024-12-31 15:00:00+08:00,test,16,1134615,2025-01-03 14:30:50.286774+08:00,2025-01-03,2025-01-03 14:31:00+08:00,RB2505.XSGE,0.838113,XSGE,RB,RB2507.XSGE,False,3276.808918,2025-07-15,2025-07-16 00:00:00+08:00,193.395251,ok,
571,75622,2024-12-31 15:00:00+08:00,test,16,1134615,2025-01-03 14:30:50.286774+08:00,2025-01-03,2025-01-03 14:31:00+08:00,RB2505.XSGE,0.838113,XSGE,RB,RB2508.XSGE,False,3278.0,2025-08-15,2025-08-16 00:00:00+08:00,224.395251,ok,
572,75622,2024-12-31 15:00:00+08:00,test,16,1134615,2025-01-03 14:30:50.286774+08:00,2025-01-03,2025-01-03 14:31:00+08:00,RB2505.XSGE,0.838113,XSGE,RB,RB2509.XSGE,False,3290.485476,2025-09-15,2025-09-16 00:00:00+08:00,255.395251,ok,
573,75622,2024-12-31 15:00:00+08:00,test,16,1134615,2025-01-03 14:30:50.286774+08:00,2025-01-03,2025-01-03 14:31:00+08:00,RB2505.XSGE,0.838113,XSGE,RB,RB2510.XSGE,False,3298.323692,2025-10-15,2025-10-16 00:00:00+08:00,285.395251,ok,
574,75622,2024-12-31 15:00:00+08:00,test,16,1134615,2025-01-03 14:30:50.286774+08:00,2025-01-03,2025-01-03 14:31:00+08:00,RB2505.XSGE,0.838113,XSGE,RB,RB2511.XSGE,False,3322.0,2025-11-17,2025-11-18 00:00:00+08:00,318.395251,ok,
575,75622,2024-12-31 15:00:00+08:00,test,16,1134615,2025-01-03 14:30:50.286774+08:00,2025-01-03,2025-01-03 14:31:00+08:00,RB2505.XSGE,0.838113,XSGE,RB,RB2512.XSGE,False,3326.0,2025-12-15,2025-12-16 00:00:00+08:00,346.395251,ok,
